## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Convolution

Convolution is a mathematical operation that is used to transform a signal, note that the signal can be 1 dimensional (a standard signal), 2 dimensional (an image), or have higher dimensions. 

The convolution operation is defined as

$$
(f * g)(t) \equiv \int_{-\infty}^{\infty} f(\tau) g(t - \tau) d\tau
$$

Where $f$ is the input signal and $g$ is the kernel (or filter). The integral computes a weighted sum of the signal $f$, where the weights are defined by the kernel $g$ flipped and shifted by $t$.

Overall, this means that when you convolve a signal (or an image, or multidimensional object) with a kernel, you obtain a new signal that has been transformed by the kernel. 

In this notebook, we will learn the basic concepts of convolution, how it is used to transform a signal in specific ways, and the connection with Convolutional Neural Networks (CNNs). 

## Learning goals
- understand what a 1D convolution does
- apply a mean smoothing filter to a signal
- see how different kernels produce different transformations
- understand stride, padding, and kernel size
- connect classical filtering to learned convolutional layers in neural networks
- build intuition for how CNNs extract useful patterns from data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#we will create a simple signal and a kernel to convolve with it
x = np.array([0, 0, 1, 3, 7, 8, 7, 3, 1, 0, 0])

plt.figure(figsize=(8, 3))
plt.plot(x, marker='o')
plt.title('Original 1D signal')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()

### What is convolution?

For a 1D signal, convolution works by:

1. placing a **kernel** (or filter), over part of the signal
2. multiplying each signal value by the corresponding filter value
3. summing the products
4. sliding the filter to the next location and repeating

In lecture 03.03 we used a median filter, which also slides a window along the signal. Convolution uses the same sliding window, but it computes a **weighted sum** instead of the median. Let us use this kernel (a 3-point moving average):

$$
kernel = \left[\frac{1}{3},\frac{1}{3},\frac{1}{3}\right]
$$

We will apply this filter to the signal one step at a time.

For example:

- signal window: [1, 3, 7]
- kernel: [1/3, 1/3, 1/3]
- product: [1/3, 1, 7/3]
- sum: 11/3

In [ ]:
x_new = np.zeros(len(x))  # space for the result
# multiply the kernel with the signal and sum the result, 
#this operation will change the first value of the signal
x_new[1] = (x[0]*(1/3)) + (x[1]*(1/3)) + (x[2]*(1/3))

x_new[2] = (x[1]*(1/3)) + (x[2]*(1/3)) + (x[3]*(1/3))

x_new[3] = (x[2]*(1/3)) + (x[3]*(1/3)) + (x[4]*(1/3))

x_new[4] = (x[3]*(1/3)) + (x[4]*(1/3)) + (x[5]*(1/3))

x_new[5] = (x[4]*(1/3)) + (x[5]*(1/3)) + (x[6]*(1/3))

x_new[6] = (x[5]*(1/3)) + (x[6]*(1/3)) + (x[7]*(1/3))

x_new[7] = (x[6]*(1/3)) + (x[7]*(1/3)) + (x[8]*(1/3))

x_new[8] = (x[7]*(1/3)) + (x[8]*(1/3)) + (x[9]*(1/3))

x_new[9] = (x[8]*(1/3)) + (x[9]*(1/3)) + (x[10]*(1/3))

#remove the first and last value of the convolved signal, since they are not valid values
x_new = x_new[1:-1]

plt.figure(figsize=(8, 3))
plt.plot(x, marker='o', label='Original Signal')
plt.plot(np.arange(1, len(x_new)+1),x_new, marker='o', label='Convolved Signal')
plt.title('Convolved 1D signal')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()
plt.show()

## What is padding?

Without padding, the output becomes shorter because the kernel cannot extend beyond the signal boundaries.

With **padding**, we add extra values around the edges so that the kernel can still be applied near the ends.

Common choices:
- no padding: output is shorter
- zero padding: add zeros at both ends
- mirror padding: mirror the signal at the extremes.

Example of zero padding 
- Original signal = `[2,1,3,7,8]`
- Padded signal = `[0,2,1,3,7,8,0]`

with padding, the output of the convolution is the same length as the original signal. 

In [ ]:
kernel = np.array([1/3, 1/3, 1/3])   # 3-point moving average

# add one zero at the start and one zero at the end of the signal
x_padded = np.pad(x, (1, 1), mode='constant')
print('Original length:', len(x), '| Padded length:', len(x_padded))

# 'valid' applies the kernel only where it fits completely inside the (padded) signal
x_new_padded = np.convolve(x_padded, kernel, mode='valid')
print('Output length:  ', len(x_new_padded))

plt.figure(figsize=(8, 3))
plt.plot(x, marker='o', label='Original Signal')
plt.plot(x_new_padded, marker='o', label='Convolved Signal with Padding')
plt.title('Convolved 1D signal with Padding')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()
plt.show()

## What is stride?

The **stride** determines how far the kernel moves each time.

Typically, the kernel moves one point at the time, this is called stride 1. However, you can adjust the stride and have the kernel applied only to portions of the signal 

- stride = 1: move one step at a time
- stride = 2: move two steps at a time
- stride = 5: move five steps at a time


Larger stride means:
- fewer output points
- more aggressive downsampling

In [ ]:
# stride 2: keep every second output (the kernel moves two points at a time)
x_new_stride = x_new_padded[::2]
idx_stride = np.arange(len(x_new_padded))[::2]   # position of each kept point in the original signal

fig, ax = plt.subplots(3, 1, figsize=(8, 9), sharex=True, sharey=True)
ax[0].plot(x, marker='o')
ax[1].plot(x_new_padded, marker='o')
ax[2].plot(idx_stride, x_new_stride, marker='s')
ax[0].set_title(f'Original 1D signal ({len(x)} points)')
ax[1].set_title(f'Convolved, stride 1 ({len(x_new_padded)} points)')
ax[2].set_title(f'Convolved, stride 2 ({len(x_new_stride)} points)')
for a in ax:
    a.set_ylabel('Amplitude')
    a.grid(True)
ax[2].set_xlabel('Index')
plt.show()

## Different kernels produce different transformations

The effect of convolution depends entirely on the kernel values.

Examples:
- smoothing kernel: averages nearby points
- difference kernel: highlights changes
- edge-like kernel: responds strongly when the signal rises or falls rapidly
- weighted smoothing kernel: gives more importance to the center point

In [ ]:
kernels = {
    'Mean smoothing': np.array([1/3, 1/3, 1/3]),
    'Weighted smoothing': np.array([0.25, 0.5, 0.25]),
    'First difference': np.array([1, -1]),
    'Edge detector': np.array([-1, 0, 1]),
    'Sharpen-like': np.array([-1, 3, -1])
}

plt.figure(figsize=(10, 10))
for i, (name, k) in enumerate(kernels.items(), start=1):
    y = np.convolve(x, k, mode='same')
    plt.subplot(len(kernels), 1, i)
    plt.plot(y, marker='o')
    plt.title(f'{name} | kernel = {k}')
    plt.grid(True)
plt.tight_layout()
plt.show()

### How to interpret these kernels

**Mean smoothing**
- reduces local noise
- preserves broad trends

**Weighted smoothing**
- still smooths the signal
- gives more influence to the center point

**First difference**
- estimates local change
- positive values indicate increase
- negative values indicate decrease

**Edge detector**
- responds to rapid changes across a wider local region
- often highlights transitions or boundaries

**Sharpen-like kernel**
- emphasizes local peaks relative to neighbors


**A note on the direction of the kernel**

The definition of convolution **flips** the kernel before it slides it along the signal. `np.convolve` does this. So the kernel `[-1, 0, 1]` gives a **negative** value when the signal goes up (look at the edge detector plot).

The convolution layers in CNNs do **not** flip the kernel (this operation is called *cross-correlation*, `np.correlate`). For a learned kernel, this is not important: the network learns the kernel in the direction that it needs.

## Other examples

In [ ]:
np.random.seed(2)
x_smooth_true = np.sin(np.linspace(0, 4*np.pi, 100))
noise = np.random.normal(scale=0.25, size=100)
x_noisy = x_smooth_true + noise

smoother = np.ones(5) / 5
y_noisy_smooth = np.convolve(x_noisy, smoother, mode='same')   # 'same': output has the same length and is aligned with the input

plt.figure(figsize=(10, 4))
plt.plot(x_noisy, label='Noisy signal', alpha=0.7)
plt.plot(x_smooth_true, label='Underlying true signal', linewidth=2)
plt.plot(y_noisy_smooth, label='Smoothed signal', linewidth=2)
plt.title('Convolution can reduce noise')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
edge = np.array([1, 0, -1])   # flipped by np.convolve, so the output is x[n+1] - x[n-1]: positive when the signal goes up
y_noisy_edge = np.convolve(x_noisy, edge, mode='same')
y_smooth_edge = np.convolve(y_noisy_smooth, edge, mode='same')

plt.figure(figsize=(10, 4))
plt.plot(x_noisy, label='Noisy signal', alpha=0.7)
plt.plot(x_smooth_true, label='Underlying true signal', linewidth=2)
plt.plot(y_noisy_edge, label='Edge kernel on the noisy signal', alpha=0.7)
plt.plot(y_smooth_edge, label='Edge kernel on the smoothed signal', linewidth=2)
plt.title('Convolution can enhance change')
plt.xlabel('Index')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()
plt.show()

The edge kernel shows where the signal goes up (positive) and down (negative). On the noisy signal, it also makes the noise larger. If we smooth first and then apply the edge kernel, the result is much cleaner. This is the idea of a CNN with **two layers**: the output of one kernel is the input of the next kernel.

## From signal processing to neural networks

So far, we **chose the kernel values ourselves**.

Examples:
- mean smoothing: `[1/3, 1/3, 1/3]`
- edge detector: `[-1, 0, 1]`

However, we used data to learn the weights $\omega_i$ of a linear model and of an artificial neural network. We can also use data to learn the best kernels.

In a **convolutional neural network**, the kernel values are not manually chosen.
Instead, they are **learned from data** during training.

That means the model discovers which local patterns are useful for solving the task.

## Learning a kernel from data

A convolution is a weighted sum of the points in a window. A linear model is also a weighted sum. So we can **learn a kernel with linear regression**:

1. Cut the input signal into windows of 3 points. Each window is one row of $X$.
2. The target $y$ is the output signal.
3. The coefficients of the linear regression are the kernel.

We make a "secret" kernel, apply it to the noisy signal, and then try to find it from the input and output signals only.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.linear_model import LinearRegression

secret_kernel = np.array([0.2, 0.5, -0.3])
x_in = x_noisy
y_out = np.correlate(x_in, secret_kernel, mode='valid')    # no flip, as in a CNN layer
y_out = y_out + np.random.normal(scale=0.02, size=y_out.size)   # a small measurement noise

X_windows = sliding_window_view(x_in, 3)     # each row: 3 consecutive points
print('X shape:', X_windows.shape, '| y shape:', y_out.shape)

lin = LinearRegression(fit_intercept=False).fit(X_windows, y_out)
print('Secret kernel: ', secret_kernel)
print('Learned kernel:', lin.coef_.round(3))

Linear regression found the kernel from the data. A CNN does the same thing, but:
- it learns **many kernels** at the same time,
- it puts a **nonlinearity** after each kernel,
- it stacks **many layers**,
- the target is not a signal: it is the class (or the value) that we want to predict.

So the CNN learns the kernels that are the most useful to predict the target. We use gradient descent, because there is no simple formula for this. In the next notebook, 05.02b, we will build a CNN with PyTorch.